# Transit Window Evaluation for MMWN Analysis

For each planet × sector in `tess_availability.csv`, this notebook:
1. Downloads TESS SPOC PDC-SAP 2-min lightcurve
2. Cleans (NaN, quality flags, sigma-clip outliers)
3. Finds the longest gap-free segment per sector
4. Estimates transit duration T14 from system parameters
5. Folds by period to find empirical T0 from data
6. Predicts individual transit midpoints in-sector
7. For each transit, extracts a 3T window (T pre + T in + T post)
8. Checks for gaps inside each window
9. Refines midpoint empirically (flux-weighted centroid)
10. Reports usable transits per planet

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightkurve as lk
from scipy.stats import median_abs_deviation
from IPython.display import display

warnings.filterwarnings('ignore')

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
FOLDER   = os.path.join(BASE_DIR, 'Final HJ and UHJ')

AV_CSV  = os.path.join(FOLDER, 'tess_availability_batch3.csv')
HJ_CSV  = os.path.join(FOLDER, 'hot_jupiters_batch3.csv')
UHJ_CSV = os.path.join(FOLDER, 'ultra_hot_jupiters_batch3.csv')

print('Setup complete — Batch 3')

## Planet Parameters

Period and star radius come from our TEPCat CSVs. Transit duration T14 is estimated from:
$$T_{14} = \frac{P}{\pi} \arcsin\!\left(\frac{R_\star}{a}\sqrt{(1+k)^2 - b^2}\right)$$
where $k = R_p/R_\star$ and $b=0$ (central transit approximation — conservative, slightly overestimates T14). 

OGLE-TR-056 is manually added since it replaced WASP-103 in our list.
You can override any T14 value in `T14_OVERRIDE_H` if you have a published value.

In [ ]:
hj_df  = pd.read_csv(HJ_CSV)
uhj_df = pd.read_csv(UHJ_CSV)

params_df = pd.concat(
    [hj_df [['Planet','Period_days','Radius_Rjup','StarRadius_Rsun','SemiMajorAxis_AU']],
     uhj_df[['Planet','Period_days','Radius_Rjup','StarRadius_Rsun','SemiMajorAxis_AU']]],
    ignore_index=True
).drop_duplicates('Planet').set_index('Planet')

R_sun = 6.957e8
R_jup = 7.149e7
AU    = 1.496e11

def estimate_T14(P_days, R_star_rsun, R_p_rjup, a_AU):
    R_s = R_star_rsun * R_sun
    R_p = R_p_rjup   * R_jup
    a   = a_AU * AU
    arg = np.clip((R_s + R_p) / a, 0, 1)
    return (P_days / np.pi) * np.arcsin(arg)

T14_OVERRIDE_H = {
    # 'PlanetName': T14_hours  — override if you have a published value
}

PLANET_PARAMS = {}
for planet, row in params_df.iterrows():
    P     = row['Period_days']
    T14_d = estimate_T14(P, row['StarRadius_Rsun'], row['Radius_Rjup'], row['SemiMajorAxis_AU'])
    T14_h = T14_d * 24.0
    if planet in T14_OVERRIDE_H and T14_OVERRIDE_H[planet] is not None:
        T14_h = T14_OVERRIDE_H[planet]
        T14_d = T14_h / 24.0
    PLANET_PARAMS[planet] = dict(P=P, T14_h=T14_h, T14_d=T14_d)

summary = pd.DataFrame([
    {'Planet': k, 'Period_days': v['P'], 'T14_h (est)': round(v['T14_h'], 2)}
    for k, v in PLANET_PARAMS.items()
])
print('Batch 3 — Estimated transit durations:')
display(summary)

## Helper Functions

In [ ]:
# ── Gap detection ─────────────────────────────────────────────────────────────

def find_continuous_segments(time, max_gap_min=5.0):
    max_gap_d = max_gap_min / 1440.0
    diffs = np.diff(time)
    gap_mask = diffs > max_gap_d
    breaks = np.where(gap_mask)[0] + 1
    starts = np.concatenate([[0], breaks])
    ends   = np.concatenate([breaks - 1, [len(time) - 1]])
    return list(zip(starts.tolist(), ends.tolist()))


def longest_segment(time, max_gap_min=5.0):
    segs = find_continuous_segments(time, max_gap_min)
    best = max(segs, key=lambda s: time[s[1]] - time[s[0]])
    return best[0], best[1], time[best[1]] - time[best[0]]


def has_gap_in_window(time, t_center, half_width_d, max_gap_min=5.0):
    mask = (time >= t_center - half_width_d) & (time <= t_center + half_width_d)
    t_win = time[mask]
    if len(t_win) < 3:
        return True
    max_gap_d = max_gap_min / 1440.0
    return bool(np.any(np.diff(t_win) > max_gap_d))


# ── Lightcurve cleaning ───────────────────────────────────────────────────────

def clean_lc(lc, sigma=3.0):
    lc = lc.remove_nans().remove_outliers(sigma=sigma)
    time = np.asarray(lc.time.value, dtype=float)
    flux = np.asarray(lc.flux.value, dtype=float)
    flux = flux / np.nanmedian(flux)
    return time, flux


# ── Empirical T0 from phase fold ──────────────────────────────────────────────

def find_empirical_T0(time, flux, period, n_phase_bins=200):
    t_ref = float(time[0])
    phase = ((time - t_ref) % period) / period
    bins  = np.linspace(0, 1, n_phase_bins + 1)
    bin_centers = 0.5 * (bins[:-1] + bins[1:])
    bin_flux = np.array([
        np.median(flux[(phase >= bins[i]) & (phase < bins[i+1])])
        if np.any((phase >= bins[i]) & (phase < bins[i+1])) else np.nan
        for i in range(n_phase_bins)
    ])
    min_idx   = np.nanargmin(bin_flux)
    min_phase = float(bin_centers[min_idx])
    T0 = t_ref + min_phase * period
    return T0, min_phase, bin_centers, bin_flux


# ── Empirical midpoint refinement with dip validation ────────────────────────

def refine_midpoint(time, flux, t_nominal, T14_d, snr_threshold=2.0, max_shift_fraction=0.5):
    """
    Starting from a predicted midpoint t_nominal:
      1. Estimate local baseline and noise from the flanking T14 windows.
      2. Check if a dip is actually detected (SNR >= snr_threshold).
      3. Find the empirical dip center via flux-weighted centroid.
      4. If the centroid shift exceeds max_shift_fraction * T14,
         do a finer local-minimum search within ±0.5*T14 of the predicted center
         and re-run the centroid from there.
      5. Return refined midpoint, shift quality flag, depth, and SNR.

    Returns
    -------
    t_refined     : float  — best-estimate midpoint (BTJD)
    depth         : float  — transit depth (fractional, relative to baseline)
    snr           : float  — depth / baseline_noise
    midpoint_note : str    — 'GOOD', 'ADJUSTED', or 'NO_DIP'
    n_in          : int    — number of in-transit points used
    """
    # ── Baseline: flanking T from predicted center ────────────────────────────
    base_mask = (
        ((time >= t_nominal - 1.5*T14_d) & (time < t_nominal - 0.5*T14_d)) |
        ((time >  t_nominal + 0.5*T14_d) & (time <= t_nominal + 1.5*T14_d))
    )
    if base_mask.sum() < 5:
        return t_nominal, np.nan, np.nan, 'NO_DIP', 0

    baseline = np.median(flux[base_mask])
    noise    = np.std(flux[base_mask])
    if noise == 0:
        noise = 1e-6

    # ── In-transit region around predicted center ─────────────────────────────
    in_mask = (time >= t_nominal - 0.6*T14_d) & (time <= t_nominal + 0.6*T14_d)
    if in_mask.sum() < 3:
        return t_nominal, np.nan, np.nan, 'NO_DIP', 0

    t_in = time[in_mask]
    f_in = flux[in_mask]

    depth = baseline - np.min(f_in)
    snr   = depth / noise

    # ── Check if dip is actually present ─────────────────────────────────────
    if snr < snr_threshold:
        # Dip not significant at predicted location — report but do not move
        return t_nominal, depth, snr, 'NO_DIP', int(in_mask.sum())

    # ── Flux-weighted centroid from predicted center ──────────────────────────
    weights = np.maximum(0, baseline - f_in)
    if weights.sum() == 0:
        return t_nominal, depth, snr, 'NO_DIP', int(in_mask.sum())

    t_centroid = float(np.sum(weights * t_in) / np.sum(weights))
    shift_frac = abs(t_centroid - t_nominal) / T14_d

    # ── If shift is large, try local minimum search first ────────────────────
    if shift_frac > max_shift_fraction:
        # Scan ±0.5*T14 around predicted for the actual flux minimum
        scan_mask = (time >= t_nominal - 0.5*T14_d) & (time <= t_nominal + 0.5*T14_d)
        if scan_mask.sum() >= 3:
            t_scan = time[scan_mask]
            f_scan = flux[scan_mask]
            # Rolling 5-point median to smooth noise before finding minimum
            smooth = np.convolve(f_scan, np.ones(5)/5, mode='same')
            local_min_idx = np.argmin(smooth)
            t_local_min   = float(t_scan[local_min_idx])

            # Re-run centroid centred on local minimum
            in2_mask = (time >= t_local_min - 0.6*T14_d) & (time <= t_local_min + 0.6*T14_d)
            if in2_mask.sum() >= 3:
                t_in2 = time[in2_mask]
                f_in2 = flux[in2_mask]
                w2    = np.maximum(0, baseline - f_in2)
                if w2.sum() > 0:
                    t_centroid = float(np.sum(w2 * t_in2) / np.sum(w2))
                    depth      = baseline - np.min(f_in2)
                    snr        = depth / noise
                    in_mask    = in2_mask

        shift_frac_final = abs(t_centroid - t_nominal) / T14_d
        note = 'ADJUSTED' if shift_frac_final < shift_frac else 'ADJUSTED'
    else:
        note = 'GOOD'

    return t_centroid, depth, snr, note, int(in_mask.sum())


# ── Window extractor ──────────────────────────────────────────────────────────

def extract_transit_window(time, flux, t_mid, T14_d, max_gap_min=5.0):
    hw   = 1.5 * T14_d
    mask = (time >= t_mid - hw) & (time <= t_mid + hw)
    t_win = time[mask]
    f_win = flux[mask]
    gap_inside = has_gap_in_window(time, t_mid, hw, max_gap_min)
    n_pre  = int(np.sum(t_win < t_mid - 0.5*T14_d))
    n_post = int(np.sum(t_win > t_mid + 0.5*T14_d))
    n_tr   = len(t_win) - n_pre - n_post
    return t_win, f_win, gap_inside, n_pre, n_tr, n_post


print('Helper functions defined.')

## Main Analysis — Planet × Sector Loop

For each planet and each available sector, we:
- Download + clean the LC
- Fold to find empirical T0
- Enumerate individual transits
- Check 3T window integrity and refine midpoints
- Flag usable transits (gap-free, ≥10 points in each of the 3 segments)

## Visual Transit Inspector\nSelect a planet from the dropdown to see all its sector lightcurves. Use this to visually confirm transit dips before running the full analysis.

In [ ]:
import math

av_df_w  = pd.read_csv(AV_CSV)
hj_df_w  = pd.read_csv(HJ_CSV)
uhj_df_w = pd.read_csv(UHJ_CSV)
meta_df  = pd.concat([hj_df_w, uhj_df_w], ignore_index=True).set_index('Planet')

_GAP_MIN = 5.0
NCOLS    = 5

planet_list = av_df_w[av_df_w['Status'] == 'Available']['Planet'].tolist()

for planet in planet_list:
    P     = PLANET_PARAMS[planet]['P']
    T14_d = PLANET_PARAMS[planet]['T14_d']
    T14_h = PLANET_PARAMS[planet]['T14_h']

    try:
        R_b   = float(meta_df.loc[planet, 'Radius_Rjup'])
        depth = float(meta_df.loc[planet, 'Transit_depth_pct'])
    except Exception:
        R_b, depth = float('nan'), float('nan')

    print(f'[{planet}] Downloading...', flush=True)
    try:
        sr = lk.search_lightcurve(planet, mission='TESS', cadence=120, author='SPOC')
        if len(sr) == 0:
            print(f'  No data — skipped.\n')
            continue
        lc_collection = sr.download_all(quality_bitmask='default')
    except Exception as e:
        print(f'  Download error: {e}\n')
        continue

    n_sectors = len(lc_collection)
    nrows     = math.ceil(n_sectors / NCOLS)
    print(f'  {n_sectors} sector(s) — plotting...', flush=True)

    fig, axes = plt.subplots(nrows, NCOLS,
                             figsize=(NCOLS * 3.8, nrows * 3.2),
                             squeeze=False)

    r_str = f'R_b={R_b:.2f} Rjup' if np.isfinite(R_b) else ''
    d_str = f'depth={depth:.3f}%'  if np.isfinite(depth) else ''
    fig.suptitle(
        f'{planet}  |  {r_str}  |  P={P:.3f} d  |  {d_str}  |  '
        f'T14≈{T14_h:.2f} hr  |  {n_sectors} sectors',
        fontsize=11, fontweight='bold', y=1.01
    )

    for idx, lc_raw in enumerate(lc_collection):
        row_i      = idx // NCOLS
        col_i      = idx  % NCOLS
        ax         = axes[row_i][col_i]
        sector_num = lc_raw.meta.get('SECTOR', '?')

        try:
            time, flux = clean_lc(lc_raw)
        except Exception as e:
            ax.set_title(f'S{sector_num} — error', fontsize=8)
            ax.axis('off')
            continue

        span_d = time[-1] - time[0]

        # Predict transit midpoints for subplot title count only
        T0_emp, _, _, _ = find_empirical_T0(time, flux, P)
        T0_emp  = float(T0_emp)
        t_start = time[0]  - 1.5 * T14_d
        t_end   = time[-1] + 1.5 * T14_d
        n_min   = int(np.ceil( (t_start - T0_emp) / P ))
        n_max   = int(np.floor((t_end   - T0_emp) / P ))
        midpoints = [float(T0_emp + n * P)
                     for n in range(n_min, n_max + 1)
                     if t_start <= T0_emp + n * P <= t_end]

        # Raw flux — no overlays so dips are visible
        ax.plot(time, flux, '.', ms=1.0, color='#c0392b', alpha=0.5, zorder=2)

        # Y-limits scaled to show transit depth clearly
        depth_frac = depth / 100.0 if np.isfinite(depth) else 0.01
        flux_min   = np.nanpercentile(flux, 1)
        flux_max   = np.nanpercentile(flux, 99)
        margin     = max(3 * depth_frac, (flux_max - flux_min) * 0.1, 0.002)
        ylo = min(flux_min, 1.0 - 2 * depth_frac) - margin * 0.3
        yhi = max(flux_max, 1.0 + depth_frac)     + margin * 0.3
        ax.set_ylim(ylo, yhi)

        ax.set_title(
            f'S{sector_num}  |  {span_d:.1f}d  |  {len(time)} pts  |  ~{len(midpoints)} transits',
            fontsize=7.5
        )
        ax.set_xlabel('BTJD', fontsize=7)
        ax.tick_params(labelsize=6.5)
        ax.yaxis.set_major_formatter(plt.FormatStrFormatter('%.3f'))

    for idx in range(n_sectors, nrows * NCOLS):
        axes[idx // NCOLS][idx % NCOLS].set_visible(False)

    plt.tight_layout()
    plt.show()
    print(f'  Done.\n', flush=True)

In [ ]:
av_df = pd.read_csv(AV_CSV)

MIN_POINTS_PER_SEGMENT = 10
MAX_GAP_MIN = 5.0
SNR_THRESHOLD = 2.0
MAX_SHIFT_FRACTION = 0.5

all_transit_records = []

# Colour cycle for continuous segments
SEG_COLORS = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336',
              '#00BCD4', '#8BC34A', '#FF5722', '#3F51B5', '#009688']

for _, av_row in av_df.iterrows():
    planet      = av_row['Planet']
    ptype       = av_row['Type']
    sectors_str = str(av_row['Sectors'])

    if av_row['Status'] != 'Available' or sectors_str.strip() in ('', 'nan'):
        print(f'\n[SKIP] {planet} — no TESS SPOC 2-min data')
        continue

    if planet not in PLANET_PARAMS:
        print(f'\n[SKIP] {planet} — no planet params found')
        continue

    P     = PLANET_PARAMS[planet]['P']
    T14_d = PLANET_PARAMS[planet]['T14_d']
    T14_h = PLANET_PARAMS[planet]['T14_h']

    print(f'\n{"="*65}')
    print(f'  {planet}  [{ptype}]  P={P:.4f}d  T14={T14_h:.2f}h')
    print(f'  Sectors: {sectors_str}')
    print(f'{"="*65}')

    print(f'  Downloading TESS SPOC 2-min lightcurves...')
    try:
        sr = lk.search_lightcurve(planet, mission='TESS', cadence=120, author='SPOC')
        if len(sr) == 0:
            print('  No data found — skipping.')
            continue
        lc_collection = sr.download_all(quality_bitmask='default')
    except Exception as e:
        print(f'  Download error: {e}')
        continue

    for lc_raw in lc_collection:
        sector_num = lc_raw.meta.get('SECTOR', '?')
        print(f'\n  --- Sector {sector_num} ---')

        try:
            time, flux = clean_lc(lc_raw)
        except Exception as e:
            print(f'    Clean failed: {e}')
            continue

        print(f'    {len(time)} clean points  |  span: {time[-1]-time[0]:.2f} days')

        # ── Identify continuous segments ──────────────────────────────────────
        segs = find_continuous_segments(time, MAX_GAP_MIN)
        seg_durs = [(time[e] - time[s], s, e) for s, e in segs]
        seg_durs_sorted = sorted(seg_durs, reverse=True)
        longest_dur, ls0, ls1 = seg_durs_sorted[0]

        print(f'    {len(segs)} continuous segment(s)  |  longest: {longest_dur:.2f} days  ({ls1-ls0+1} pts)')

        # ── Sector lightcurve plot ────────────────────────────────────────────
        fig, axes = plt.subplots(2, 1, figsize=(15, 6), sharex=True,
                                 gridspec_kw={'height_ratios': [3, 1]})
        fig.suptitle(f'{planet}  —  Sector {sector_num}  |  '
                     f'P={P:.4f}d  T14={T14_h:.2f}h  |  '
                     f'{len(segs)} segment(s), longest={longest_dur:.2f}d',
                     fontsize=11)

        ax   = axes[0]   # lightcurve panel
        ax_g = axes[1]   # gap/segment map panel

        # Plot gaps as faint grey background first
        ax.plot(time, flux, '.', ms=1, color='#cccccc', alpha=0.4, zorder=1)

        for i, (s, e) in enumerate(segs):
            col   = SEG_COLORS[i % len(SEG_COLORS)]
            label = f'Seg {i+1} ({time[e]-time[s]:.2f}d)'
            # Mark longest segment with a star in the label
            if s == ls0 and e == ls1:
                label += ' ★ longest'
            ax.plot(time[s:e+1], flux[s:e+1], '.', ms=2,
                    color=col, alpha=0.8, zorder=2, label=label)
            # Segment bar on gap map panel
            ax_g.barh(0, time[e] - time[s], left=time[s],
                      height=0.6, color=col, alpha=0.85)

        # Shade gap regions on main panel
        gap_edges = []
        for i in range(len(segs) - 1):
            gap_start = time[segs[i][1]]
            gap_end   = time[segs[i+1][0]]
            ax.axvspan(gap_start, gap_end, color='red', alpha=0.08, zorder=0)
            gap_edges.append((gap_start, gap_end))

        ax.set_ylabel('Normalized PDC-SAP flux')
        ax.legend(fontsize=7, loc='lower right', ncol=min(4, len(segs)),
                  markerscale=4, framealpha=0.7)
        ax.set_ylim(np.nanpercentile(flux, 0.2) - 0.002,
                    np.nanpercentile(flux, 99.8) + 0.002)

        ax_g.set_yticks([])
        ax_g.set_xlabel('Time (BTJD)')
        ax_g.set_ylabel('Segments', fontsize=8)
        ax_g.set_xlim(time[0] - 0.3, time[-1] + 0.3)
        # Mark gaps on segment bar panel
        for gs, ge in gap_edges:
            ax_g.axvspan(gs, ge, color='red', alpha=0.25)

        plt.tight_layout()
        plt.show()

        # ── Continue with analysis ────────────────────────────────────────────
        s0_l, s1_l, seg_dur = longest_segment(time, MAX_GAP_MIN)

        T0_emp, min_phase, ph_bins, ph_flux = find_empirical_T0(time, flux, P)
        T0_emp = float(T0_emp)
        print(f'    Empirical T0 (BTJD): {T0_emp:.6f}  (phase of min = {min_phase:.4f})')

        t_start = time[0]  - 1.5 * T14_d
        t_end   = time[-1] + 1.5 * T14_d
        n_min   = int(np.ceil( (t_start - T0_emp) / P ))
        n_max   = int(np.floor((t_end   - T0_emp) / P ))
        predicted_midpoints = [
            float(T0_emp + n * P) for n in range(n_min, n_max + 1)
            if t_start <= T0_emp + n * P <= t_end
        ]
        print(f'    Predicted transits in sector: {len(predicted_midpoints)}')
        print(f'    {"#":>3}  {"t_pred (BTJD)":>14}  {"t_refined":>14}  {"shift":>7}  '
              f'{"SNR":>5}  {"depth":>7}  {"pre":>4}  {"in":>4}  {"post":>4}  {"mid":^8}  status')

        transit_num = 0
        for t_pred in predicted_midpoints:
            t_refined, depth, snr, mid_note, n_in = refine_midpoint(
                time, flux, t_pred, T14_d,
                snr_threshold=SNR_THRESHOLD,
                max_shift_fraction=MAX_SHIFT_FRACTION,
            )

            t_win, f_win, gap_inside, n_pre, n_tr, n_post = extract_transit_window(
                time, flux, t_refined, T14_d, MAX_GAP_MIN
            )

            shift_min = float((t_refined - t_pred) * 1440.0)

            enough_points = (n_pre  >= MIN_POINTS_PER_SEGMENT and
                             n_tr   >= MIN_POINTS_PER_SEGMENT and
                             n_post >= MIN_POINTS_PER_SEGMENT)

            if mid_note == 'NO_DIP':
                status = 'NO_DIP'
            elif gap_inside:
                status = 'GAP'
            elif not enough_points:
                status = 'FEW_PTS'
            else:
                status = 'OK'

            transit_num += 1
            snr_s   = f'{snr:.1f}'   if np.isfinite(snr)   else ' N/A'
            depth_s = f'{depth:.4f}' if np.isfinite(depth) else '  N/A'
            print(f'    {transit_num:>3}  {t_pred:>14.6f}  {t_refined:>14.6f}  '
                  f'{shift_min:>+6.1f}m  {snr_s:>5}  {depth_s:>7}  '
                  f'{n_pre:>4}  {n_tr:>4}  {n_post:>4}  {mid_note:^8}  [{status}]')

            all_transit_records.append({
                'planet':        planet,
                'type':          ptype,
                'sector':        sector_num,
                'transit_num':   transit_num,
                't_predicted':   round(t_pred,    6),
                't_refined':     round(t_refined, 6),
                'shift_min':     round(shift_min, 2),
                'n_pre':         n_pre,
                'n_transit':     n_tr,
                'n_post':        n_post,
                'depth_est':     round(float(depth), 6) if np.isfinite(depth) else float('nan'),
                'snr':           round(float(snr),   2) if np.isfinite(snr)   else float('nan'),
                'midpoint_note': mid_note,
                'gap_inside':    gap_inside,
                'status':        status,
                'T14_h_used':    round(T14_h, 3),
                'T0_empirical':  round(T0_emp, 6),
            })

print('\n\nLoop complete.')

## Summary Table

In [ ]:
rec_df = pd.DataFrame(all_transit_records)

if rec_df.empty:
    print('No records — re-run the loop cell.')
else:
    print(f'Total transit windows evaluated: {len(rec_df)}')
    print(f'Usable (OK):  {(rec_df.status=="OK").sum()}')
    print(f'Has gap:      {(rec_df.status=="GAP").sum()}')
    print(f'Too few pts:  {(rec_df.status=="FEW_PTS").sum()}')
    print()

    # Per-planet summary
    grp = rec_df.groupby(['planet','type']).agg(
        total_transits=('transit_num','count'),
        ok=('status', lambda x: (x=='OK').sum()),
        T14_h=('T14_h_used','first'),
        T0_emp=('T0_empirical','first'),
        mean_shift_min=('shift_min','mean'),
    ).reset_index().round({'mean_shift_min':2})

    print('Per-planet summary:')
    display(grp)

    # Full table
    print('\nFull transit record:')
    display(rec_df)

## Save Results

In [ ]:
out_csv  = os.path.join(FOLDER, 'transit_windows_evaluated_batch3.csv')
out_xlsx = os.path.join(FOLDER, 'transit_windows_evaluated_batch3.xlsx')

rec_df.to_csv(out_csv, index=False)

with pd.ExcelWriter(out_xlsx, engine='openpyxl') as writer:
    rec_df.to_excel(writer, sheet_name='All Transits', index=False)
    ok_df = rec_df[rec_df.status == 'OK']
    ok_df.to_excel(writer, sheet_name='OK Only', index=False)
    grp.to_excel(writer, sheet_name='Planet Summary', index=False)

print(f'Saved:\n  {out_csv}\n  {out_xlsx}')

## Diagnostic Plots — Phase-Folded LC per Planet

Shows the folded lightcurve used to determine T0, and overplots all individual OK transit windows.

In [ ]:
# Re-download one sector per planet (the first OK sector) for diagnostic plot
# This cell is optional — run after the main loop

ok_df = rec_df[rec_df.status == 'OK'].copy()

for planet in ok_df['planet'].unique():
    prows = ok_df[ok_df.planet == planet]
    P     = PLANET_PARAMS[planet]['P']
    T14_d = PLANET_PARAMS[planet]['T14_d']
    T0    = prows['T0_empirical'].iloc[0]

    try:
        sr = lk.search_lightcurve(planet, mission='TESS', cadence=120, author='SPOC')
        lc_all = sr.download_all(quality_bitmask='default').stitch()
        time, flux = clean_lc(lc_all)
    except Exception as e:
        print(f'{planet}: download error — {e}')
        continue

    # Phase fold
    phase = ((time - T0) % P) / P
    phase[phase > 0.5] -= 1.0   # centre transit at phase=0

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(f'{planet}  |  P={P:.4f}d  T14={T14_d*24:.2f}h', fontsize=12)

    # Left: phase fold
    ax = axes[0]
    ax.plot(phase * P * 24, flux, '.', ms=1.5, alpha=0.3, color='steelblue')
    ax.set_xlabel('Time from mid-transit (h)')
    ax.set_ylabel('Normalized flux')
    ax.set_title('Phase-folded LC')
    ax.set_xlim(-2*T14_d*24, 2*T14_d*24)
    # Mark T14 edges
    for edge in [-0.5*T14_d*24, 0.5*T14_d*24]:
        ax.axvline(edge, color='orange', lw=1, ls='--', label='T14 edge')
    ax.axvline(0, color='red', lw=1, ls=':', label='T0')
    ax.legend(fontsize=7, loc='upper right')

    # Right: midpoint shifts histogram
    ax2 = axes[1]
    shifts = prows['shift_min'].values
    ax2.hist(shifts, bins=min(20, max(5, len(shifts))), color='steelblue', edgecolor='white')
    ax2.axvline(0, color='red', lw=1, ls=':')
    ax2.set_xlabel('Midpoint shift from ephemeris (min)')
    ax2.set_ylabel('Count')
    ax2.set_title(f'Empirical mid-transit shifts  (N={len(shifts)} OK transits)')

    plt.tight_layout()
    plt.show()
    print(f'  {planet}: {len(prows)} OK transits  |  mean shift = {shifts.mean():.2f} min  std = {shifts.std():.2f} min\n')